# Tool 3: Remap the hypnogram labels

This tool harmonizes the sleep stage labels across a heterogeneous PSG database, converting the various scoring
conventions to the standard AASM labels (`W`, `N1`, `N2`, `N3`, `R`).

- **Inputs**: a study folder holding the `.edf` recordings and, next to each, its hypnogram exported as text (one
  label per 30 s epoch, e.g. `<recording>_Hypnogram_Export.txt`).
- **Outputs**: the remapped hypnograms, written next to the originals with the output suffix
  (`_hypno_remapped.txt` by default, a study keeps the name it already has); `mid_uncertain_epochs_to_verify.tsv`
  (your corrections of the reviewed epochs), `config_param/file_names.json` (the suffixes read and written, followed
  by the later tools) and, when you keep non-AASM stages, `config_param/custom_stages.json`, in the study folder.
- **Sections**: 1. Study folder (scan the hypnograms), 2. Review uncertain and unexpected epochs, 3. Define remapping
  rules, 4. Preview and save remapped hypnograms, 5. Verify remapping.

In [ ]:
try:
    import os
    import re
    import json
    import numpy as np
    import pandas as pd
    from pathlib import Path
    import ipywidgets as widgets
    from types import SimpleNamespace
    from ipyfilechooser import FileChooser
    from IPython.display import display, HTML
except ImportError as e:
    print('❌ Import error, a package is missing:', e)

# Shared EDF reading (tools/edf_io_lib.py, also used by tool 0), probed from the repo root (the canonical launch
# folder) or from tools/, so the import works whatever folder Voila is launched from.
import sys
for _p in [os.getcwd(), os.path.join(os.getcwd(), 'tools'),
           os.path.join(os.path.dirname(os.getcwd()), 'tools'), os.path.join(os.getcwd(), '..', 'tools')]:
    if os.path.isfile(os.path.join(_p, 'edf_io_lib.py')):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        break
from edf_io_lib import load_custom_stages, parse_custom_field, mid_recording_unscored
import ui_lib as UI   # the look shared by the Voila pages
import study_files_lib as SF   # the names registry and the choice of the companion files

#%% --- CONSTANTS ---

STANDARD_LABELS   = {'W', 'N1', 'N2', 'N3', 'R'}
ACCEPTABLE_LABELS = STANDARD_LABELS | {'MT'}
STANDARD_OPTIONS  = ['W', 'N1', 'N2', 'N3', 'R', 'MT', '']

DEFAULT_MAPPING = {
    'W': 'W', 'N1': 'N1', 'N2': 'N2', 'N3': 'N3', 'R': 'R',
    '0': 'W', '1': 'N1', '2': 'N2', '3': 'N3', '4': 'N3', '5': 'R',
    '?': 'W',
    'MT': 'MT',
    'S1': 'N1', 'S2': 'N2', 'S3': 'N3', 'S4': 'N3',
    'WAKE': 'W', 'REM': 'R', 'SWS': 'N3',
}

#%% --- STATE ---

STATE = SimpleNamespace(
    folder_path=None,
    hypno_suffix=None,
    hypno_files=[],
    hypno_data={},
    failed_load=[],
    label_configs=None,
    config_labels=None,
    configs=None,
    suspicious_labels=None,
    mid_question_files=[],
    unexpected_epochs={},
    corrections={},
    remap_by_config=None,
    question_flags=None,
    output_suffix=SF.HYPNO_REMAPPED_DEFAULT,
    tsv_conflicts=[],
    tsv_loaded_count=0,
)
STATE.ui = {}

#%% --- HELPER FUNCTIONS ---

def detect_hypno_files(folder, suffix):
    found = []
    for root, _, files in os.walk(folder):
        for fname in files:
            if fname.startswith('.'):
                continue
            if suffix:
                if os.path.normcase(suffix) in os.path.normcase(fname):
                    found.append(os.path.join(root, fname))
            else:
                if fname.lower().endswith('.txt'):
                    found.append(os.path.join(root, fname))
    return sorted(found)


def extract_file_id(filepath, suffix):
    stem = Path(filepath).stem
    if suffix:
        suffix_stem = Path(suffix).stem
        if os.path.normcase(stem).endswith(os.path.normcase(suffix_stem)):
            return stem[:-len(suffix_stem)]
    return stem


def apply_remap(hypno, remap_dict):
    remapped = hypno.copy()
    for raw_label, new_label in remap_dict.items():
        if new_label:
            remapped[remapped == raw_label] = new_label
    return remapped


def get_unexpected_labels(unique_labels):
    return [l for l in unique_labels if l not in STANDARD_LABELS]


def suggest_label(raw_label):
    return DEFAULT_MAPPING.get(raw_label, DEFAULT_MAPPING.get(raw_label.upper(), ''))


def is_suspicious_label(label):
    # Any label not in DEFAULT_MAPPING is unrecognised and should be flagged
    return label not in DEFAULT_MAPPING


def save_custom_stages(folder, new_stages):
    """Merge new_stages into config_param/custom_stages.json (existing kept in order, new appended). Returns merged list."""
    path = Path(folder) / 'config_param' / 'custom_stages.json'
    path.parent.mkdir(parents=True, exist_ok=True)
    merged = list(load_custom_stages(folder))
    for s in new_stages:
        if s and s not in merged:
            merged.append(s)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump({'custom_stages': merged}, f, indent=2, ensure_ascii=False)
    return merged


def current_acceptable():
    """AASM labels + MT + the user-declared custom (non-AASM) stages, read live from the Section 1 field."""
    return STANDARD_LABELS | {'MT'} | set(parse_custom_field(custom_stages_box.value))


def get_epoch_context(hypno, epoch_idx, n_ctx=3):
    start = max(0, epoch_idx - n_ctx)
    end   = min(len(hypno), epoch_idx + n_ctx + 1)
    parts = []
    for i in range(start, end):
        parts.append(f"[{hypno[i]}]" if i == epoch_idx else str(hypno[i]))
    return " ".join(parts)


def print_in_scrollable_box(text, height=250, font_size="12px"):
    display(HTML(
        f'<pre style="overflow-y:scroll; height:{height}px; border:1px solid #ccc; '
        f'padding:8px; font-size:{font_size};">{text}</pre>'
    ))


def write_uncertain_tsv(folder, hypno_data, question_flags, unexpected_epochs,
                        corrections=None):
    """Write mid-'?' and unexpected-label epochs to a unified TSV for manual verification."""
    rows = []
    for file_id, (idxs, n_total) in question_flags.items():
        hypno = hypno_data.get(file_id)
        if hypno is None:
            continue
        for epoch_idx in idxs:
            ctx       = get_epoch_context(hypno, epoch_idx, n_ctx=5)
            corrected = (corrections or {}).get(file_id, {}).get(epoch_idx, '')
            rows.append({
                'participant_id':  file_id,
                'epoch_index':     epoch_idx,
                'epoch_time_sec':  epoch_idx * 30,
                'total_epochs':    n_total,
                'original_label':  '?',
                'context':         ctx,
                'corrected_label': corrected,
            })
    for label, files in (unexpected_epochs or {}).items():
        for file_id, epoch_list in files.items():
            hypno = hypno_data.get(file_id)
            if hypno is None:
                continue
            n_total = len(hypno)
            for epoch_idx in epoch_list:
                ctx       = get_epoch_context(hypno, epoch_idx, n_ctx=5)
                corrected = (corrections or {}).get(file_id, {}).get(epoch_idx, '')
                rows.append({
                    'participant_id':  file_id,
                    'epoch_index':     epoch_idx,
                    'epoch_time_sec':  epoch_idx * 30,
                    'total_epochs':    n_total,
                    'original_label':  label,
                    'context':         ctx,
                    'corrected_label': corrected,
                })
    rows.sort(key=lambda r: (r['participant_id'], r['epoch_index']))
    df = pd.DataFrame(rows, columns=[
        'participant_id', 'epoch_index', 'epoch_time_sec',
        'total_epochs', 'original_label', 'context', 'corrected_label',
    ])
    tsv_path = Path(folder) / 'mid_uncertain_epochs_to_verify.tsv'
    df.to_csv(tsv_path, sep='\t', index=False)
    return tsv_path


def merge_uncertain_tsv(folder, hypno_data, question_flags, unexpected_epochs,
                        corrections=None):
    """Update the TSV by merging corrections into the existing file.

    Preserves all existing rows and their original_label. Updates corrected_label
    where a correction exists in memory. Adds new rows for uncertain epochs not yet
    in the file. Falls back to write_uncertain_tsv if the file does not exist yet.
    """
    tsv_path = Path(folder) / 'mid_uncertain_epochs_to_verify.tsv'

    # Build combined correction lookup: {(fid, epoch_idx): corrected_label}
    all_corrections = {}
    for fid, corrs in (corrections or {}).items():
        for ep, lbl in corrs.items():
            all_corrections[(fid, int(ep))] = lbl

    if not tsv_path.exists():
        return write_uncertain_tsv(
            folder, hypno_data, question_flags, unexpected_epochs,
            corrections=corrections,
        )

    try:
        df = pd.read_csv(tsv_path, sep='\t', dtype=str).fillna('')
    except Exception:
        df = pd.DataFrame(columns=[
            'participant_id', 'epoch_index', 'epoch_time_sec',
            'total_epochs', 'original_label', 'context', 'corrected_label',
        ])

    # Update corrected_label for existing rows
    for idx, row in df.iterrows():
        try:
            key = (str(row['participant_id']), int(row['epoch_index']))
        except (ValueError, TypeError):
            continue
        if key in all_corrections:
            df.at[idx, 'corrected_label'] = all_corrections[key]

    # Collect epochs already in the TSV
    existing_keys = set()
    for _, row in df.iterrows():
        try:
            existing_keys.add((str(row['participant_id']), int(row['epoch_index'])))
        except (ValueError, TypeError):
            pass

    # Add new rows for uncertain epochs not yet in the TSV
    new_rows = []
    for file_id, (idxs, n_total) in question_flags.items():
        hypno = hypno_data.get(file_id)
        if hypno is None:
            continue
        for epoch_idx in idxs:
            if (file_id, epoch_idx) not in existing_keys:
                ctx = get_epoch_context(hypno, epoch_idx, n_ctx=5)
                new_rows.append({
                    'participant_id':  file_id,
                    'epoch_index':     epoch_idx,
                    'epoch_time_sec':  epoch_idx * 30,
                    'total_epochs':    n_total,
                    'original_label':  '?',
                    'context':         ctx,
                    'corrected_label': all_corrections.get((file_id, epoch_idx), ''),
                })
    for label, files in (unexpected_epochs or {}).items():
        for file_id, epoch_list in files.items():
            hypno = hypno_data.get(file_id)
            if hypno is None:
                continue
            n_total = len(hypno)
            for epoch_idx in epoch_list:
                if (file_id, epoch_idx) not in existing_keys:
                    ctx = get_epoch_context(hypno, epoch_idx, n_ctx=5)
                    new_rows.append({
                        'participant_id':  file_id,
                        'epoch_index':     epoch_idx,
                        'epoch_time_sec':  epoch_idx * 30,
                        'total_epochs':    n_total,
                        'original_label':  label,
                        'context':         ctx,
                        'corrected_label': all_corrections.get((file_id, epoch_idx), ''),
                    })

    if new_rows:
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

    df['epoch_index'] = pd.to_numeric(df['epoch_index'], errors='coerce')
    df = df.sort_values(['participant_id', 'epoch_index']).reset_index(drop=True)
    df.to_csv(tsv_path, sep='\t', index=False)
    return tsv_path


def load_uncertain_tsv(tsv_path, hypno_data):
    """Load corrections from an existing mid_uncertain_epochs_to_verify.tsv.

    Compares each row's original_label against the current hypnogram content.
    Rows whose label changed on disk (re-scoring) are returned as conflicts, not applied.
    """
    try:
        df = pd.read_csv(tsv_path, sep='\t', dtype=str).fillna('')
    except Exception:
        return {}, [], 0

    required_cols = {'participant_id', 'epoch_index', 'original_label', 'corrected_label'}
    if not required_cols.issubset(df.columns):
        return {}, [], 0

    corrections   = {}
    conflicts     = []
    loaded_count  = 0

    for _, row in df.iterrows():
        fid           = str(row['participant_id'])
        corrected_lbl = str(row['corrected_label']).strip()
        original_lbl  = str(row['original_label']).strip()

        if not corrected_lbl or fid not in hypno_data:
            continue

        try:
            epoch_idx = int(row['epoch_index'])
        except (ValueError, TypeError):
            continue

        hypno = hypno_data[fid]
        if epoch_idx < 0 or epoch_idx >= len(hypno):
            continue

        current_lbl = str(hypno[epoch_idx])

        # Conflict: hypnogram was re-scored since TSV was written
        if current_lbl != original_lbl:
            conflicts.append({
                'participant_id':  fid,
                'epoch_index':     epoch_idx,
                'original_label':  original_lbl,
                'current_label':   current_lbl,
                'corrected_label': corrected_lbl,
            })
            continue

        corrections.setdefault(fid, {})[epoch_idx] = corrected_lbl
        loaded_count += 1

    return corrections, conflicts, loaded_count


def _show_btn_in(out_widget, btn_key):
    btn = STATE.ui.get(btn_key)
    try:
        out_widget.clear_output()
        if btn is not None:
            with out_widget:
                display(btn)
    except Exception:
        pass


# What a section shows until the step it needs is done (the sections build themselves, as in tool 4).
def _show_waiting(out_widget, section):
    out_widget.clear_output()
    with out_widget:
        display(HTML(UI.waiting_html(section)))


# A choice changed upstream: forget what depends on it and put the sections after it back on hold.
def invalidate_after(stage):
    if stage == 'scan':
        STATE.corrections = {}
        for _out in (out_tsv_info, out_review, out_proceed_s2, out_proceed_s2_msg):
            _out.clear_output()
    if stage in ('scan', 'question'):
        STATE.remap_by_config = None
        out_remap.clear_output()
        with out_remap:
            display(HTML(UI.msg('muted', 'Click <b>Save corrections and continue</b> in section 2 first.',
                                symbol=False)))
    if stage in ('scan', 'question', 'remap'):
        _show_waiting(out_save, 3)
        out_end.clear_output()
    if stage in ('scan', 'question', 'remap', 'save'):
        _show_btn_in(out_verify, 'run_verify')    # a real button: it reloads the files written on disk

#%% --- SECTION HEADERS ---

section1 = widgets.HTML(UI.section_html(1, 'Study folder', '''
<p>Select the folder containing your database. The tool will automatically detect all
hypnogram files matching the specified suffix.
<br>&#x2022; Choose your study folder using the file browser below.
<br>&#x2022; The hypnogram suffix is auto-detected from the selected folder and pre-filled
    (e.g. <code>_Hypnogram_Export.txt</code>). Adjust manually if needed.
<br>&#x2022; The <b>Advanced</b> box holds the output suffix of the remapped files
    (<code>_hypno_remapped.txt</code> for a new study; a study keeps the name it already has).
<br>&#x2022; Leave the hypnogram suffix empty to detect all <code>.txt</code> files.
<br>&#x2022; Check the box to skip participants whose remapped file already exists.
<br>&#x2022; <b>Custom stages</b> &#x2014; if your project intentionally keeps non-AASM labels (e.g. <code>N4</code>, a movement stage), list them in the <b>Custom stages</b> field below (comma-separated). They are preserved by the remapping, accepted by the verification, and recognised by tools 6/7/8. Non-AASM targets can also be registered from Section 3.
<br>&#x2022; Click <b>Scan</b> to detect the files and their label configurations: section 2 opens when some
    epochs need a review, otherwise section 3 appears directly.</p>
'''))

section2 = widgets.HTML(UI.section_html(2, 'Review uncertain and unexpected epochs', '''
<p>This section reviews two kinds of problematic epochs found during the scan, in a single widget that opens
after the scan when there are some:
<br>&#x2022; <b>Mid-recording <code>?</code> epochs</b> &#x2014; unscored epochs in the middle of the
    recording (outside the first/last 10 epochs). Verify these directly in your sleep scoring
    software before assigning a label.
<br>&#x2022; <b>Unexpected labels</b> &#x2014; labels not recognised in the default mapping
    (e.g. <code>U</code>, <code>M</code>). Inspect the surrounding context to understand each
    label, then assign a standard replacement.
<br>&#x2022; Use the <b>Show</b> filter to focus on mid-<code>?</code> or one specific label, and
    <b>Apply to all in this file</b> to batch-replace a label across the entire current
    participant in a single click.
<br>&#x2022; <b>TSV manual corrections</b> &#x2014; if a <code>mid_uncertain_epochs_to_verify.tsv</code>
    file already exists in the study folder (e.g. edited manually between sessions or after
    re-scoring), it is loaded automatically at scan time and the corrections are pre-filled in
    the review widget (shown in green, still editable). A summary is shown below. The TSV is
    never overwritten during scanning &#x2014; only when you click <b>Corrections done</b>.
<br>Use the review widget as needed, then click <b>Save corrections and continue</b>: the corrections are
    written to the TSV and section 3 appears. Click it as well when every epoch was already corrected in an earlier
    session.</p>
'''))

section3 = widgets.HTML(UI.section_html(3, 'Define remapping rules', '''
<p>For each label configuration found in your database, assign the target standard AASM label.
<br>&#x2022; Click a configuration to unfold its list of original labels.
<br>&#x2022; Use the dropdowns to assign a standard label
    (<code>W</code>, <code>N1</code>, <code>N2</code>, <code>N3</code>, <code>R</code>).
    Suggestions are pre-filled.
<br>&#x2022; Suspicious labels are highlighted in red with &#xb1;5 epochs of surrounding context.
<br>&#x2022; If <code>?</code> labels appear here, they are boundary epochs (first/last ~10 epochs)
    not reviewed in Section 2 &#x2014; map them to <code>W</code>.
<br>&#x2022; Repeat for each configuration, then click <b>Save remapping</b>: section 4 appears.</p>
'''))

section4 = widgets.HTML(UI.section_html(4, 'Preview and save remapped hypnograms', '''
<p>&#x2022; The output suffix is set in Section 1 (Advanced box).
<br>&#x2022; Click <b>Preview</b> to inspect the remapping before saving.
<br>&#x2022; Click <b>Save files</b> to write the remapped hypnograms next to the originals.
<br>If unexpected labels remain after remapping, a confirmation will be required before saving.</p>
'''))

section5 = widgets.HTML(UI.section_html(5, 'Verify remapping', '''
<p>Click <b>Verify</b> to reload the saved files and check that the remapping was applied correctly across
the full database.</p>
'''))

#%% --- OUTPUT ZONES ---

out_scan           = widgets.Output()
out_configs        = widgets.Output()
out_tsv_info       = widgets.Output()
out_review         = widgets.Output()
out_proceed_s2     = widgets.Output()
out_proceed_s2_msg = widgets.Output()
out_remap          = widgets.Output()
out_save           = widgets.Output()
out_verify         = widgets.Output()
out_end            = widgets.Output()

#%% --- SECTION 1: FOLDER SELECTION & SCAN ---

chooser = FileChooser(os.getcwd())
chooser.title = UI.picker_title('Study folder')
chooser.show_only_dirs = True

suffix_box = widgets.Text(
    value='_Hypnogram_Export.txt',
    description='Hypnogram suffix:',
    placeholder='e.g. _Hypnogram_Export.txt  (leave empty for all .txt)',
    style=UI.LABEL_STYLE,
    layout=widgets.Layout(width=UI.FIELD_WIDTH)
)

suffix_info = widgets.HTML(value='')
# a menu of the detected suffixes in front of the suffix field (the field stays the value the scan reads)
suffix_menu, fill_suffix_menu = UI.suffix_menu(suffix_box, suffix_info)

output_suffix_box = widgets.Text(
    value=SF.HYPNO_REMAPPED_DEFAULT,
    description='Output suffix:',
    placeholder='e.g. _hypno_remapped.txt',
    style=UI.LABEL_STYLE,
    layout=widgets.Layout(width=UI.FIELD_WIDTH)
)

# the output suffix proposed for this study (a study keeps its name): the default of the Advanced box's field
_proposed_output = {'suffix': SF.HYPNO_REMAPPED_DEFAULT}
advanced, mark_advanced = UI.advanced_box([output_suffix_box])


def _update_advanced_mark(change=None):
    """'⚠️ default changed' in the title of the Advanced box when the output suffix differs from the one proposed."""
    mark_advanced(['output suffix'] if output_suffix_box.value.strip() != _proposed_output['suffix'] else [])


output_suffix_box.observe(_update_advanced_mark, names='value')

exclude_existing_check = widgets.Checkbox(
    value=False,
    description='Skip participants already remapped',
    layout=widgets.Layout(width='380px')
)

custom_stages_box = widgets.Text(
    value='',
    description='Custom stages:',
    placeholder='e.g. N4, Mvt  (non-AASM labels to keep: leave empty for none)',
    style=UI.LABEL_STYLE,
    layout=widgets.Layout(width=UI.FIELD_WIDTH)
)

custom_stages_info = widgets.HTML(value='')

btn_scan = UI.button('Scan', 'read')
scan_progress, scan_progress_row = UI.progress_bar('Files:')


def _detect_hypno_suffixes(chooser):
    """Auto-detect hypnogram .txt suffixes in the selected folder and pre-fill suffix_box."""
    fill_suffix_menu({}, [])   # emptied until the detection below
    if not chooser.selected:
        suffix_info.value = ''
        return
    data_folder = Path(chooser.selected)
    # Pre-fill the 'Custom stages' field from any custom_stages.json already saved for this dataset.
    _existing_custom = load_custom_stages(chooser.selected)
    custom_stages_box.value = ', '.join(_existing_custom)
    custom_stages_info.value = (
        (f'<small style="color:{UI.OK_COLOR};">Loaded custom stage(s) from config_param/custom_stages.json: '
         f'{", ".join(_existing_custom)}</small>') if _existing_custom else
        f'<small style="color:{UI.MUTED_COLOR};">No custom (non-AASM) stages registered for this dataset yet.</small>'
    )
    edf_files = [f for f in sorted(data_folder.rglob('*')) if f.suffix.lower() == '.edf' and not f.name.startswith('._')]
    n_total = len(edf_files)
    if n_total == 0:
        suffix_info.value = (
            f'<small style="color:{UI.WARN_COLOR};">No EDF files found in selected folder '
            '(recursive scan).</small>'
        )
        return
    # Every .txt named <recording name><suffix>, counted by suffix (a participant can have both a raw and a
    # remapped hypnogram).
    all_txt = [f for f in data_folder.rglob('*') if f.suffix.lower() == '.txt']
    suffix_owners = SF.companion_suffix_owners(edf_files, all_txt)
    suffix_counts = {s: len(r) for s, r in suffix_owners.items()}
    names, names_warn = SF.load_names(data_folder)
    # a study keeps its names: the output suffix tool 3 wrote before, else the short default
    _proposed_output['suffix'] = SF.output_suffix(names, 'hypnogram_remapped', suffix_counts,
                                                  SF.HYPNO_REMAPPED_DEFAULT, SF.HYPNO_REMAPPED_FORMER)
    output_suffix_box.value = _proposed_output['suffix']
    _update_advanced_mark()
    if not suffix_counts:
        suffix_info.value = (
            f'<small style="color:{UI.WARN_COLOR};">No hypnogram files detected — '
            'check that hypnograms are present or adjust the suffix manually.</small>'
        )
    else:
        # Pre-choice (study_files_lib.pick_hypno_suffix, intent 'scored'): the raw export of the scoring software,
        # i.e. the input name tool 3 recorded, else the most frequent suffix, the shortest on ties, setting aside the
        # event / summary / study-log exports, the remapped hypnograms (this tool's output included) and the
        # hypnograms the toolkit writes (tool 3bis, tool 0's rescoring). Set aside from the PRE-CHOICE only: every
        # .txt suffix stays in the menu, so a wrong detection stays visible and correctable by hand.
        best_suffix, best_count, _reason = SF.pick_hypno_suffix(suffix_counts, 'scored', names,
                                                                own_outputs=[output_suffix_box.value])
        # nothing to pre-choose: back to the default (an empty field would read EVERY .txt file)
        suffix_box.value = best_suffix or '_Hypnogram_Export.txt'
        _notes = ''.join(f'<br><small>{UI.msg(k, t)}</small>'
                         for k, t in SF.choice_notes(_reason, 'scored', names_warn))
        fill_suffix_menu(suffix_owners, [f.stem for f in edf_files], _notes)


chooser.register_callback(_detect_hypno_suffixes)


def on_scan(_=None):
    out_scan.clear_output()
    out_configs.clear_output()
    out_tsv_info.clear_output()

    folder = chooser.selected_path
    if not folder:
        with out_scan:
            print("⚠️ Please select a folder first.")
        return

    suffix       = suffix_box.value.strip()
    remap_suffix = output_suffix_box.value.strip()
    STATE.folder_path   = folder
    STATE.hypno_suffix  = suffix
    STATE.output_suffix = remap_suffix

    with out_scan:
        print(f"📁 Folder: {folder}")
        if suffix:
            print(f"🔍 Hypnogram suffix: '{suffix}'")
        else:
            print("🔍 No suffix filter (all .txt files)")
        if remap_suffix:
            print(f"💾 Output suffix: '{remap_suffix}'")

        all_raw_files = detect_hypno_files(folder, suffix)

        if not all_raw_files:
            print("\n⚠️ No hypnogram files found. Check your suffix or folder.")
            return

        print(f"\n✅ Found {len(all_raw_files)} hypnogram file(s).")

        hypno_data  = {}
        failed      = []
        output      = ""
        dyn_out     = widgets.Output()
        display(dyn_out)

        scan_progress.max = max(1, len(all_raw_files))
        scan_progress.value = 0
        scan_progress.bar_style = 'info'
        for fpath in all_raw_files:
            file_id  = extract_file_id(fpath, suffix)
            output  += f"Loading: {file_id}\n"
            with dyn_out:
                dyn_out.clear_output(wait=True)
                print_in_scrollable_box(output)
            try:
                hypno = np.loadtxt(fpath, dtype=str).astype('<U10')
                hypno_data[file_id] = hypno
            except Exception as e:
                output += f"  ❌ Error: {e}\n"
                failed.append((fpath, str(e)))
            scan_progress.value += 1
        UI.finish_progress(scan_progress, n_failed=len(failed))

        dyn_out.clear_output()

        STATE.failed_load = failed
        print(f"Loaded: {len(hypno_data)}  |  Failed: {len(failed)}")

        # --- Check for existing remapped files ---
        existing_remap     = []
        existing_remap_ids = set()
        if remap_suffix:
            existing_remap     = detect_hypno_files(folder, remap_suffix)
            existing_remap_ids = {extract_file_id(f, remap_suffix) for f in existing_remap}

            if existing_remap:
                print(f"\n⚠️ {len(existing_remap)} remapped hypnogram(s) already exist "
                      f"with suffix '{remap_suffix}'.")
            else:
                print(f"\n✅ No remapped files found yet with suffix '{remap_suffix}'.")

            if exclude_existing_check.value and existing_remap_ids:
                n_before     = len(all_raw_files)
                active_files = [f for f in all_raw_files
                                if extract_file_id(f, suffix) not in existing_remap_ids]
                active_data  = {k: v for k, v in hypno_data.items()
                                if k not in existing_remap_ids}
                n_excluded   = n_before - len(active_files)
                print(f"   ℹ️ {n_excluded} participant(s) skipped (already remapped). "
                      f"Processing {len(active_files)} remaining.")
            else:
                active_files = all_raw_files
                active_data  = hypno_data
        else:
            active_files = all_raw_files
            active_data  = hypno_data

        # --- Warn about .txt files matching neither suffix ---
        if suffix and remap_suffix:
            all_txt   = detect_hypno_files(folder, '')
            known_set = {os.path.normcase(p) for p in set(all_raw_files) | set(existing_remap)}
            other_txt = [f for f in all_txt if os.path.normcase(f) not in known_set]
            if other_txt:
                print(f"\nℹ️ {len(other_txt)} .txt file(s) found that match neither suffix:")
                for f in other_txt[:5]:
                    print(f"   {Path(f).name}")
                if len(other_txt) > 5:
                    print(f"   … and {len(other_txt) - 5} more.")

        # --- Load existing uncertain TSV if present ---
        # Corrections are loaded here but applied to the data only AFTER the flags below are
        # computed, so TSV-corrected epochs still appear in Section 2 (pre-filled) instead of
        # vanishing from the review widget.
        tsv_path    = Path(folder) / 'mid_uncertain_epochs_to_verify.tsv'
        tsv_existed = tsv_path.exists()
        if tsv_existed:
            corr_tsv, tsv_conflicts, tsv_loaded = load_uncertain_tsv(
                tsv_path, active_data
            )
            if tsv_loaded > 0:
                print(f"\n📋 Existing TSV loaded: {tsv_loaded} correction(s) pre-applied in memory.")
            if tsv_conflicts:
                print(f"  ⚠️ {len(tsv_conflicts)} conflict(s) detected "
                      f"(label changed since TSV was written): see Section 2.")
        else:
            corr_tsv, tsv_conflicts, tsv_loaded = {}, [], 0

        # Snapshot the ORIGINAL labels before applying TSV corrections, so the flags below
        # (question_flags / unexpected_epochs) are computed on the raw scoring; TSV-corrected
        # epochs then appear in Section 2 pre-filled (green) rather than disappearing.
        original_data = active_data
        if corr_tsv:
            original_data = {fid: arr.copy() for fid, arr in active_data.items()}
            for fid, corrs in corr_tsv.items():
                for epoch_idx, lbl in corrs.items():
                    if fid in active_data:
                        active_data[fid][epoch_idx] = lbl

        # --- Set STATE from active (filtered + pre-corrected) data ---
        STATE.hypno_files = active_files
        STATE.hypno_data  = active_data

        # --- Compute question_flags on ORIGINAL labels (before TSV corrections), so epochs
        #     already corrected via the TSV still appear in Section 2 (pre-filled). ---
        question_flags = {}
        for file_id, hypno in original_data.items():
            problematic = mid_recording_unscored(hypno)        # the '?' more than 10 epochs from both ends
            if problematic:
                question_flags[file_id] = (problematic, len(hypno))

        # --- Label configurations ---
        configs_dict = {}
        for file_id, hypno in active_data.items():
            key = frozenset(np.unique(hypno).tolist())
            configs_dict.setdefault(key, [])
            configs_dict[key].append(file_id)

        configs       = list(configs_dict.keys())
        config_labels = [
            f"config. {i+1} (n={len(configs_dict[c])})"
            for i, c in enumerate(configs)
        ]

        STATE.label_configs = configs_dict
        STATE.configs       = configs
        STATE.config_labels = config_labels

        if active_data:
            print(f"\n📊 {len(configs)} unique label configuration(s) found:")
            for i, cfg in enumerate(configs):
                print(f"  Config. {i+1} (n={len(configs_dict[cfg])}): {sorted(cfg)}")

        # --- Suspicious labels (computed on ORIGINAL labels, see note above) ---
        # Labels declared as custom (non-AASM) in Section 1 are NOT treated as suspicious:
        # they are split into `custom_found` and only reported informationally below, so they
        # never reach Section 2 or the uncertain TSV.
        custom_now   = parse_custom_field(custom_stages_box.value)
        suspicious   = {}
        custom_found = {}
        for file_id, hypno in original_data.items():
            for label in np.unique(hypno):
                if not is_suspicious_label(label):
                    continue
                if label in custom_now:
                    custom_found.setdefault(label, [])
                    custom_found[label].append(file_id)
                else:
                    suspicious.setdefault(label, [])
                    suspicious[label].append(file_id)
        STATE.suspicious_labels = suspicious

        # --- Unexpected epochs (all occurrences of each suspicious label, by file) ---
        unexpected_epochs = {}
        for label, file_ids in suspicious.items():
            unexpected_epochs[label] = {}
            for fid in file_ids:
                epochs = [int(i) for i in np.where(original_data[fid] == label)[0]]
                unexpected_epochs[label][fid] = epochs
        STATE.unexpected_epochs = unexpected_epochs

        # --- Mid-recording '?' ---
        STATE.mid_question_files = list(question_flags.keys())
        STATE.question_flags     = question_flags

        # --- Combined report: uncertain ('?') + unexpected labels ---
        # '?' (mid-recording) is treated like any other flagged label, so the two former
        # messages are merged into a single aligned list (one row per label, listing the
        # affected file names instead of an example context).
        issue_rows = []  # (label, files, n_epochs)
        if question_flags:
            q_files = list(question_flags.keys())
            q_eps   = sum(len(idxs) for idxs, _ in question_flags.values())
            issue_rows.append(('?', q_files, q_eps))
        for label in sorted(suspicious.keys()):
            files = sorted(suspicious[label])
            n_eps = sum(len(unexpected_epochs[label][f]) for f in files)
            issue_rows.append((label, files, n_eps))

        if issue_rows:
            total_eps = sum(n for _, _, n in issue_rows)
            print(f"\n⚠️ Uncertain / unexpected labels detected "
                  f"({len(issue_rows)} label(s), {total_eps} epoch(s) total):")
            # Pad the quoted label so the 'N file(s)' column lines up for multi-char labels.
            w = max(len(f"'{lbl}'") for lbl, _, _ in issue_rows)
            for label, files, n_eps in issue_rows:
                qlabel = f"'{label}'".ljust(w)
                shown  = ', '.join(files[:8])
                if len(files) > 8:
                    shown += f", … (+{len(files) - 8} more)"
                print(f"  {qlabel} : {len(files)} file(s), {n_eps} epoch(s), {shown}")
            print("  → Review and assign mappings in Section 2, or verify them in your "
                  "scoring software.")
        else:
            print("\n✅ No uncertain or unexpected labels detected.")

        # --- Custom (non-AASM) stages declared in Section 1 and actually present ---
        if custom_found:
            cw = max(len(f"'{lbl}'") for lbl in custom_found)
            print("\nℹ️ Custom (non-AASM) stage(s) declared in Section 1 and detected:")
            for label in sorted(custom_found):
                files  = sorted(set(custom_found[label]))
                n_eps  = sum(int(np.count_nonzero(original_data[f] == label)) for f in files)
                qlabel = f"'{label}'".ljust(cw)
                shown  = ', '.join(files[:8])
                if len(files) > 8:
                    shown += f", … (+{len(files) - 8} more)"
                print(f"  {qlabel} : {len(files)} file(s), {n_eps} epoch(s), {shown}")
            print("  → Preserved by the remapping and recognised by tools 6/7/8 "
                  "(rename them in Section 3 if needed).")

        # --- Write TSV only if it didn't already exist ---
        has_uncertain = bool(question_flags or suspicious)
        if not tsv_existed and has_uncertain:
            new_tsv_path = write_uncertain_tsv(
                folder, active_data, question_flags, unexpected_epochs
            )
            print(f"  → TSV exported for manual review:\n     {new_tsv_path}")
        elif tsv_existed:
            print(f"  ℹ️ TSV already exists, not overwritten at scan time.")

        # --- Configs summary table ---
        max_len = max((len(sorted(c)) for c in configs), default=0)
        data = {}
        for i, cfg in enumerate(configs):
            fids      = configs_dict[cfg]
            col       = f"config. {i+1}<br>(n={len(fids)})"
            ls        = sorted(cfg)
            data[col] = ls + [''] * (max_len - len(ls))

        if data:
            df_table = pd.DataFrame(data)
            df_table.index = pd.Index(range(1, max_len + 1), name='label #')
            with out_configs:
                out_configs.clear_output()
                display(HTML("<h4>Label configurations found:</h4>"))
                display(HTML(df_table.to_html(escape=False)))

        invalidate_after('scan')

        # Restore corrections from TSV (invalidate_after reset them to {})
        STATE.corrections      = corr_tsv
        STATE.tsv_conflicts    = tsv_conflicts
        STATE.tsv_loaded_count = tsv_loaded

        has_issues = bool(STATE.mid_question_files or STATE.unexpected_epochs)

        # --- TSV info panel shown at top of Section 2 ---
        with out_tsv_info:
            out_tsv_info.clear_output()
            if tsv_existed and (tsv_loaded > 0 or tsv_conflicts):
                if tsv_loaded > 0:
                    per_fid = {}
                    for fid, corrs in corr_tsv.items():
                        per_fid.setdefault(fid, [])
                        for ep, lbl in sorted(corrs.items()):
                            # original_data holds the pre-correction labels (snapshot taken
                            # before the TSV was applied), so show old→new in the summary.
                            old = original_data[fid][ep] if fid in original_data else '?'
                            per_fid[fid].append(f"ep.{ep}: {old}→{lbl}")
                    summary_rows = ''.join(
                        f"<tr><td style='padding:2px 10px'>{fid}</td>"
                        f"<td style='padding:2px 10px'>{', '.join(eps)}</td></tr>"
                        for fid, eps in sorted(per_fid.items())
                    )
                    display(HTML(
                        f'<div style="{UI.box_style("ok")}">'
                        f'<b>✅ {tsv_loaded} correction(s) pre-loaded from '
                        f'<code>mid_uncertain_epochs_to_verify.tsv</code> and applied in memory.</b>'
                        f'<table style="margin-top:6px; font-size:12px;">'
                        f'<tr><th style="padding:2px 10px; text-align:left;">Participant</th>'
                        f'<th style="padding:2px 10px; text-align:left;">Corrections</th></tr>'
                        f'{summary_rows}</table></div>'
                    ))
                if tsv_conflicts:
                    conflict_rows = ''.join(
                        f"<tr><td style='padding:2px 10px'>{c['participant_id']}</td>"
                        f"<td style='padding:2px 10px; text-align:center'>{c['epoch_index']}</td>"
                        f"<td style='padding:2px 10px; text-align:center; color:{UI.WARN_COLOR}'>"
                        f"{c['original_label']}</td>"
                        f"<td style='padding:2px 10px; text-align:center; color:{UI.ERROR_COLOR}; "
                        f"font-weight:bold'>{c['current_label']}</td>"
                        f"<td style='padding:2px 10px; text-align:center'>"
                        f"{c['corrected_label']}</td></tr>"
                        for c in tsv_conflicts
                    )
                    display(HTML(
                        f'<div style="{UI.box_style("warn")}">'
                        f'<b>⚠️ {len(tsv_conflicts)} conflict(s) not applied</b> — '
                        f'the label changed in the hypnogram file since the TSV was written.<br>'
                        f'<small>These corrections were ignored. Review and re-apply manually '
                        f'in Section 2 if needed.</small>'
                        f'<table style="margin-top:8px; font-size:12px; border-collapse:collapse;">'
                        f'<tr style="background:#ffe082;">'
                        f'<th style="padding:3px 10px; text-align:left;">Participant</th>'
                        f'<th style="padding:3px 10px;">Epoch</th>'
                        f'<th style="padding:3px 10px;">TSV label</th>'
                        f'<th style="padding:3px 10px;">Current label</th>'
                        f'<th style="padding:3px 10px;">TSV correction</th>'
                        f'</tr>{conflict_rows}</table></div>'
                    ))

        # Section 2: the review widget opens directly (or a status message when nothing to review)
        if has_issues:
            run_review_widget()
        else:
            with out_review:
                out_review.clear_output()
                display(HTML(UI.msg('ok', "No mid-recording '?' or unexpected labels found: nothing to "
                                          "review.")))

        # Shared proceed button: show only when there are issues to address
        if has_issues:
            with out_proceed_s2:
                out_proceed_s2.clear_output()
                display(STATE.ui['proceed_s2'])
        else:
            with out_proceed_s2:
                out_proceed_s2.clear_output()
        with out_proceed_s2_msg:
            out_proceed_s2_msg.clear_output()
        if not has_issues:
            run_remap_widget()   # nothing to review: section 3 builds itself


btn_scan.on_click(on_scan)

#%% --- SECTION 2: REVIEW UNCERTAIN & UNEXPECTED EPOCHS ---

def run_review_widget(_=None):
    with out_review:
        out_review.clear_output()

        # Flat list of flagged epochs from both (disjoint) sources: mid-recording '?'
        # (question_flags) and unexpected labels (unexpected_epochs).
        flagged = []  # (reason, label, fid, epoch)
        for fid in STATE.mid_question_files:
            for ep in STATE.question_flags[fid][0]:
                flagged.append(('mid_question', '?', fid, ep))
        for label in sorted(STATE.unexpected_epochs.keys()):
            for fid in sorted(STATE.unexpected_epochs[label].keys()):
                for ep in STATE.unexpected_epochs[label][fid]:
                    flagged.append(('unexpected', label, fid, ep))

        if not flagged:
            display(HTML(UI.msg('ok', 'Nothing to review.')))
            return

        # Flagged epochs per file, for highlighting the context table.
        flagged_epochs_by_fid = {}
        for _, _, fid, ep in flagged:
            flagged_epochs_by_fid.setdefault(fid, set()).add(ep)

        ctx_slider = widgets.IntSlider(
            value=5, min=1, max=20, step=1,
            description='Context (epochs):',
            style={'description_width': '140px'},
            layout=widgets.Layout(width='380px'),
            continuous_update=False
        )

        # Filter: all issues, mid-'?' only, or one specific unexpected label.
        unique_labels  = sorted(STATE.unexpected_epochs.keys())
        filter_options = [('All issues', None)]
        if STATE.mid_question_files:
            filter_options.append(("Mid-recording '?'", '__mid__'))
        filter_options += [(lbl, lbl) for lbl in unique_labels]
        filter_dd = widgets.Dropdown(
            options=filter_options, value=None,
            description='Show:',
            style={'description_width': '60px'},
            layout=widgets.Layout(width='320px')
        )

        btn_prev    = widgets.Button(description='← Previous', layout=widgets.Layout(width='120px'))
        btn_next    = widgets.Button(description='Next →',     layout=widgets.Layout(width='120px'))
        nav_label   = widgets.HTML()
        context_out = widgets.Output()

        label_combo = widgets.Combobox(
            options=STANDARD_OPTIONS, value='',
            placeholder='Pick or type label…',
            ensure_option=False,
            layout=widgets.Layout(width='120px')
        )
        btn_apply = UI.button('Apply to this epoch', 'save', icon='check')
        btn_apply_all = UI.button('Apply to all in this file', 'caution')
        apply_out    = widgets.Output()
        progress_lbl = widgets.HTML()

        nav = {'idx': 0}

        def get_filtered():
            filt = filter_dd.value
            if filt is None:
                return flagged
            if filt == '__mid__':
                return [item for item in flagged if item[0] == 'mid_question']
            return [item for item in flagged if item[1] == filt]

        def update_progress():
            filtered  = get_filtered()
            corrected = sum(
                1 for (_, _, f, e) in filtered
                if e in STATE.corrections.get(f, {})
            )
            progress_lbl.value = f"<b>Progress:</b> {corrected} / {len(filtered)} corrected"

        def refresh_context():
            context_out.clear_output()
            filtered = get_filtered()
            if not filtered or nav['idx'] >= len(filtered):
                with context_out:
                    display(HTML("<p>No epochs to show.</p>"))
                nav_label.value = ""
                update_progress()
                return

            reason, label, fid, epoch = filtered[nav['idx']]
            n_total = len(STATE.hypno_data[fid])
            kind = "mid-recording '?'" if reason == 'mid_question' else 'unexpected label'
            nav_label.value = (
                f"<b><code style='color:{UI.ERROR_COLOR}'>{label}</code></b> ({kind}) — "
                f"File: <b>{fid}</b>, Epoch <b>{epoch}</b> / {n_total}"
                f"&nbsp; [{nav['idx'] + 1} / {len(filtered)}]"
            )

            hypno = STATE.hypno_data[fid]
            n_ctx = ctx_slider.value
            start = max(0, epoch - n_ctx)
            end   = min(len(hypno), epoch + n_ctx + 1)
            flagged_here = flagged_epochs_by_fid.get(fid, set())
            corr_here    = STATE.corrections.get(fid, {})

            rows = []
            for i in range(start, end):
                lbl     = hypno[i]
                is_curr = (i == epoch)
                is_corr = (i in corr_here)
                if is_curr:
                    style  = f'color:{UI.ERROR_COLOR}; font-weight:bold;'
                    prefix = '>>> '
                    sfx    = ' <<<'
                elif is_corr:
                    style  = f'color:{UI.OK_COLOR};'
                    prefix = '    '
                    sfx    = ' ✓'
                elif i in flagged_here:
                    style  = f'color:{UI.WARN_COLOR};'
                    prefix = '    '
                    sfx    = ''
                else:
                    style  = ''
                    prefix = '    '
                    sfx    = ''
                rows.append(
                    f'<tr style="{style}">'
                    f'<td style="padding:1px 14px; font-family:monospace;">{prefix}{i}</td>'
                    f'<td style="padding:1px 14px; font-family:monospace;">{lbl}{sfx}</td></tr>'
                )

            table = (
                '<table style="font-size:13px; border:1px solid #ddd; border-radius:4px;">'
                '<tr><th style="padding:2px 14px; text-align:left;">Epoch</th>'
                '<th style="padding:2px 14px; text-align:left;">Label</th></tr>'
                + ''.join(rows) + '</table>'
            )
            with context_out:
                display(HTML(table))

            existing = corr_here.get(epoch, '')
            # Mid-recording '?' are unlikely to be Wake; do not pre-suggest a label so the
            # user actively picks from the full list. Unexpected labels keep their suggestion.
            suggestion = suggest_label(label) if reason != 'mid_question' else ''
            label_combo.value = existing or suggestion
            update_progress()

        def on_prev(_=None):
            if nav['idx'] > 0:
                nav['idx'] -= 1
                refresh_context()

        def on_next(_=None):
            filtered = get_filtered()
            if nav['idx'] < len(filtered) - 1:
                nav['idx'] += 1
                refresh_context()

        def on_filter(change):
            nav['idx'] = 0
            refresh_context()

        def on_apply(_=None):
            apply_out.clear_output()
            new_label = label_combo.value.strip()
            if not new_label:
                with apply_out:
                    print("⚠️ Please select or type a label first.")
                return
            filtered = get_filtered()
            if nav['idx'] >= len(filtered):
                return
            reason, label, fid, epoch = filtered[nav['idx']]
            # `label` is the original flagged label (from the scan-time snapshot); use it as the
            # old label so the message stays correct even when the epoch was already pre-corrected
            # from an existing TSV (where hypno_data already holds the new label).
            STATE.hypno_data[fid][epoch] = new_label
            STATE.corrections.setdefault(fid, {})[epoch] = new_label
            with apply_out:
                print(f"✅ ep.{epoch}: '{label}' → '{new_label}'")
            if nav['idx'] < len(filtered) - 1:
                nav['idx'] += 1
            refresh_context()

        def on_apply_all(_=None):
            apply_out.clear_output()
            new_label = label_combo.value.strip()
            if not new_label:
                with apply_out:
                    print("⚠️ Please select or type a label first.")
                return
            filtered = get_filtered()
            if nav['idx'] >= len(filtered):
                return
            _, cur_label, cur_fid, _ = filtered[nav['idx']]
            count = 0
            for (reason, lbl, fid, epoch_idx) in flagged:
                if lbl == cur_label and fid == cur_fid:
                    STATE.hypno_data[fid][epoch_idx] = new_label
                    STATE.corrections.setdefault(fid, {})[epoch_idx] = new_label
                    count += 1
            with apply_out:
                print(f"✅ Applied '{new_label}' to all {count} occurrence(s) of "
                      f"'{cur_label}' in {cur_fid}.")
            refresh_context()

        btn_prev.on_click(on_prev)
        btn_next.on_click(on_next)
        btn_apply.on_click(on_apply)
        btn_apply_all.on_click(on_apply_all)
        filter_dd.observe(on_filter, names='value')
        ctx_slider.observe(lambda _: refresh_context(), names='value')

        display(
            ctx_slider, filter_dd, nav_label,
            widgets.HBox([btn_prev, btn_next]),
            context_out,
            widgets.HBox([label_combo, btn_apply, btn_apply_all]),
            apply_out, progress_lbl,
        )
        refresh_context()


#%% --- SECTION 2: SHARED PROCEED BUTTON ---

btn_proceed_s2 = UI.button('Save corrections and continue', 'save')
STATE.ui['proceed_s2'] = btn_proceed_s2


def on_proceed_s2(_=None):
    out_proceed_s2_msg.clear_output()
    invalidate_after('question')
    total = sum(len(v) for v in STATE.corrections.values())
    with out_proceed_s2_msg:
        if total:
            print(f"✅ {total} correction(s) applied to data in memory.")
        else:
            print("No corrections made.")
        print("   Section 3 is ready below.")
    # Merge corrections into the TSV (preserves existing rows and manual edits)
    if STATE.folder_path and (STATE.question_flags or STATE.unexpected_epochs
                              or STATE.corrections):
        try:
            tsv_path = merge_uncertain_tsv(
                STATE.folder_path,
                STATE.hypno_data,
                STATE.question_flags or {},
                STATE.unexpected_epochs,
                corrections=STATE.corrections,
            )
            with out_proceed_s2_msg:
                print(f"   TSV updated with corrections:\n   {tsv_path}")
        except Exception as e:
            with out_proceed_s2_msg:
                print(f"   ⚠️ Could not write TSV: {e}")
    run_remap_widget()   # section 3 builds itself from the reviewed hypnograms


btn_proceed_s2.on_click(on_proceed_s2)

#%% --- SECTION 3: REMAPPING WIDGET ---

def run_remap_widget(_=None):
    with out_remap:
        out_remap.clear_output()

        if not STATE.hypno_data:
            print("⚠️ Please scan your folder first (Section 1).")
            return

        configs_dict = {}
        for file_id, hypno in STATE.hypno_data.items():
            key = frozenset(np.unique(hypno).tolist())
            configs_dict.setdefault(key, [])
            configs_dict[key].append(file_id)

        configs       = list(configs_dict.keys())
        config_labels = [
            f"config. {i+1} (n={len(configs_dict[c])})"
            for i, c in enumerate(configs)
        ]

        # Declared custom (non-AASM) stages: kept by default (suggested = itself), not flagged
        # red, and offered in the dropdown so they can be picked or renamed.
        custom_now    = parse_custom_field(custom_stages_box.value)
        combo_options = STANDARD_OPTIONS + [s for s in custom_now if s not in STANDARD_OPTIONS]

        row_widgets_by_cfg = {}
        panels = []

        for cfg, cfg_label in zip(configs, config_labels):
            row_widgets_by_cfg[cfg_label] = {}
            sorted_labels = sorted(cfg)
            participants  = sorted(configs_dict[cfg])

            rows = []
            for raw_label in sorted_labels:
                is_custom = raw_label in custom_now
                if is_custom:
                    suggested  = raw_label   # keep the custom label by default; user may rename
                    is_suspect = False
                else:
                    suggested  = suggest_label(raw_label)
                    is_suspect = is_suspicious_label(raw_label)

                combo = widgets.Combobox(
                    options=combo_options, value=suggested,
                    placeholder='Pick or type…',
                    ensure_option=False,
                    layout=widgets.Layout(width='120px')
                )
                color = UI.ERROR_COLOR if is_suspect else 'inherit'
                lbl   = widgets.HTML(
                    f'<span style="font-family:monospace; color:{color}; '
                    f'width:80px; display:inline-block;">{raw_label}</span>'
                )
                arrow     = widgets.Label('→', layout=widgets.Layout(width='25px'))
                row_items = [lbl, arrow, combo]

                if is_suspect:
                    ctx_text = ''
                    for fid in sorted(configs_dict[cfg])[:1]:
                        hypno = STATE.hypno_data[fid]
                        idxs  = np.where(hypno == raw_label)[0]
                        if len(idxs) > 0:
                            idx      = int(idxs[0])
                            ctx_str  = get_epoch_context(hypno, idx, n_ctx=5)
                            ctx_text = (f"⚠️ Suspicious: context "
                                        f"({fid}, ep.{idx}): {ctx_str}")
                    row_items.append(widgets.HTML(
                        f'<small style="color:{UI.ERROR_COLOR}; margin-left:8px;">{ctx_text}</small>'
                    ))
                elif is_custom:
                    row_items.append(widgets.HTML(
                        UI.text_block(f'<small style="color:{UI.INFO_COLOR}; margin-left:8px;">ℹ️ Custom stage — '
                        f"kept as '{raw_label}' by default; type a new label to rename.</small>")
                    ))

                rows.append(widgets.HBox(row_items))
                row_widgets_by_cfg[cfg_label][raw_label] = combo

            panel_items = [
                widgets.HTML(
                    f"<i>Participants ({len(participants)}): {', '.join(participants)}</i>"
                )
            ]

            if '?' in sorted_labels:
                panel_items.append(widgets.HTML(
                    UI.text_block(f'<div style="{UI.box_style("info")}">'
                    'ℹ️ <b>Note:</b> <code>?</code> labels at this stage are boundary '
                    'epochs (first/last ~10 epochs of the recording) not reviewed in Section 2. '
                    'They are typically unscored: map them to <code>W</code>.</div>')
                ))

            panel_items.append(widgets.VBox(rows, layout=widgets.Layout(
                max_height='320px', overflow='auto',
                border='1px solid #ddd', padding='6px'
            )))

            panels.append(widgets.VBox(panel_items))

        acc = widgets.Accordion(children=panels)
        for i, lbl in enumerate(config_labels):
            acc.set_title(i, lbl)

        btn_save_remap = UI.button('Save remapping', 'save')
        save_remap_out = widgets.Output()

        def on_save_remap(_=None):
            remap = {}
            for cfg_label in config_labels:
                remap[cfg_label] = {
                    raw: combo.value.strip()
                    for raw, combo in row_widgets_by_cfg[cfg_label].items()
                }

            STATE.remap_by_config = remap
            invalidate_after('remap')
            # Non-AASM labels actually produced by the mapping (what will be IN the output).
            # Split into already-declared custom stages (kept) vs not-yet-declared.
            cur_custom   = parse_custom_field(custom_stages_box.value)
            out_custom   = sorted({v for mapping in remap.values() for v in mapping.values()
                                   if v and v not in ACCEPTABLE_LABELS})
            registered   = [l for l in out_custom if l in cur_custom]
            unregistered = [l for l in out_custom if l not in cur_custom]

            with save_remap_out:
                save_remap_out.clear_output()
                print("✅ Remapping saved:")
                for cfg_label, mapping in remap.items():
                    print(f"\n  {cfg_label}:")
                    for raw, new in mapping.items():
                        flag = "  ⚠️ empty target!" if not new else ""
                        print(f"    '{raw}'  →  '{new}'{flag}")
                if registered:
                    print(f"\nℹ️ Custom (non-AASM) stage(s) kept in the remapped output: {registered}")
                    print("  → Recognised by tools 6/7/8; verification will pass.")
                if unregistered:
                    print(f"\n⚠️ Non-AASM target label(s) not yet declared as custom: {unregistered}")
                    print("  → Standard AASM labels: W, N1, N2, N3, R  (MT also accepted).")
                    print("  → If these are intentional custom stages, register them with the")
                    print("    button below so tools 6/7/8 recognise them and verification passes.")
                    btn_register = UI.button(
                        f'Register as custom stage(s): {", ".join(unregistered)}', 'caution', icon='plus')
                    def on_register(_=None, _labels=unregistered):
                        merged = parse_custom_field(custom_stages_box.value)
                        for s in _labels:
                            if s not in merged:
                                merged.append(s)
                        custom_stages_box.value = ', '.join(merged)
                        with save_remap_out:
                            print(f"\n✅ Registered: {_labels}. Re-click 'Save remapping' to clear "
                                  "the warning.")
                    btn_register.on_click(on_register)
                    display(btn_register)
            run_save_widget()   # section 4 builds itself from the saved remapping

        btn_save_remap.on_click(on_save_remap)
        display(acc, widgets.HBox([btn_save_remap]), save_remap_out)


#%% --- SECTION 4: PREVIEW & SAVE ---

def run_save_widget(_=None):
    with out_save:
        out_save.clear_output()

        if STATE.remap_by_config is None:
            print("⚠️ Please define and save the remapping first (Section 3).")
            return

        cur_configs_dict = {}
        for file_id, hypno in STATE.hypno_data.items():
            key = frozenset(np.unique(hypno).tolist())
            cur_configs_dict.setdefault(key, [])
            cur_configs_dict[key].append(file_id)
        cur_configs = list(cur_configs_dict.keys())
        cur_labels  = [
            f"config. {i+1} (n={len(cur_configs_dict[c])})"
            for i, c in enumerate(cur_configs)
        ]
        file_id_to_cfg = {}
        for cfg, lbl in zip(cur_configs, cur_labels):
            for fid in cur_configs_dict[cfg]:
                file_id_to_cfg[fid] = lbl

        custom_now         = parse_custom_field(custom_stages_box.value)
        remapped_data      = {}
        unexpected_by_file = {}   # genuine non-AASM, non-custom labels (block save)
        custom_by_file     = {}   # declared custom stages present in the output (kept)
        for file_id, hypno in STATE.hypno_data.items():
            cfg_label = file_id_to_cfg.get(file_id)
            if cfg_label is None or cfg_label not in STATE.remap_by_config:
                continue
            remapped = apply_remap(hypno, STATE.remap_by_config[cfg_label])
            remapped_data[file_id] = remapped
            # MT is acceptable; declared custom stages are kept (info); the rest is a problem.
            non_std = [l for l in np.unique(remapped).tolist() if l not in ACCEPTABLE_LABELS]
            cust    = [l for l in non_std if l in custom_now]
            bad     = [l for l in non_std if l not in custom_now]
            if cust:
                custom_by_file[file_id] = cust
            if bad:
                unexpected_by_file[file_id] = bad

        if custom_by_file:
            kept = sorted({l for labels in custom_by_file.values() for l in labels})
            display(HTML(
                f'<div style="{UI.box_style("info")}">'
                f'<b>ℹ️ Custom (non-AASM) stage(s) kept:</b> {kept}, present in '
                f'{len(custom_by_file)} file(s). Recognised by tools 6/7/8.</div>'
            ))

        has_unexpected = bool(unexpected_by_file)
        if has_unexpected:
            display(HTML(
                f'<div style="{UI.box_style("warn")}">'
                f'<b>⚠️ Warning:</b> {len(unexpected_by_file)} file(s) still have '
                'non-standard labels after remapping.</div>'
            ))
            for fid, labels in unexpected_by_file.items():
                display(HTML(
                    f"<p style='margin:2px 0 2px 16px'>&#x2022; <b>{fid}</b>: {labels}</p>"
                ))
            display(HTML(
                "<p>Go back to Section 3 to add rules, or confirm saving below.</p>"
            ))

        display(HTML(
            f"<p><b>Output suffix:</b> <code>{STATE.output_suffix}</code> "
            f"(defined in Section 1)</p>"
        ))
        btn_preview    = UI.button('Preview', 'read', icon='eye')
        btn_save_files = UI.button('Save files', 'save', disabled=has_unexpected)
        confirm_check  = widgets.Checkbox(
            value=False,
            description='Confirm saving despite the warnings above',
            layout=widgets.Layout(width='440px')
        )

        if has_unexpected:
            def on_confirm(change):
                btn_save_files.disabled = not change['new']
            confirm_check.observe(on_confirm, names='value')
            display(confirm_check)

        preview_out    = widgets.Output()
        save_files_out = widgets.Output()
        display(widgets.HBox([btn_preview, btn_save_files]))
        display(preview_out, save_files_out)

        def on_preview(_=None):
            preview_out.clear_output()
            with preview_out:
                out_suffix = STATE.output_suffix
                print(f"Output suffix: '{out_suffix}'")
                print("\nRemapping rules:")
                for cfg_label, mapping in STATE.remap_by_config.items():
                    print(f"  {cfg_label}: {mapping}")
                print("\nLabel distribution after remapping (first 5 files):")
                for file_id, remapped in list(remapped_data.items())[:5]:
                    unique, counts = np.unique(remapped, return_counts=True)
                    dist = {k: int(v) for k, v in zip(unique.tolist(), counts.tolist())}
                    print(f"  {file_id}: {dist}")
                if len(remapped_data) > 5:
                    print(f"  ... and {len(remapped_data) - 5} more file(s).")

        def on_save_files(_=None):
            out_suffix = STATE.output_suffix
            if not out_suffix:
                with save_files_out:
                    print("⚠️ Please define an output suffix in Section 1.")
                return

            saved, failed_save = [], []

            for fpath in STATE.hypno_files:
                file_id = extract_file_id(fpath, STATE.hypno_suffix)
                if file_id not in remapped_data:
                    continue
                out_path = Path(fpath).parent / (file_id + out_suffix)
                try:
                    np.savetxt(out_path, remapped_data[file_id], fmt='%s')
                    saved.append(str(out_path))
                except Exception as e:
                    failed_save.append((str(out_path), str(e)))

            with save_files_out:
                save_files_out.clear_output()
                print(f"✅ Saved {len(saved)} remapped hypnogram(s).")
                if saved:
                    print(f"  → Location: {Path(saved[0]).parent}")
                if failed_save:
                    print(f"❌ Failed to save {len(failed_save)} file(s):")
                    for p, e in failed_save:
                        print(f"  {p}: {e}")

            # Persist any declared custom stages that actually survive in the remapped output,
            # so the downstream tools can auto-load them (config_param/custom_stages.json).
            _present_custom = sorted({
                s for s in parse_custom_field(custom_stages_box.value)
                if any(s in set(np.unique(r).tolist()) for r in remapped_data.values())
            })
            if _present_custom and STATE.folder_path:
                try:
                    _merged = save_custom_stages(STATE.folder_path, _present_custom)
                    with save_files_out:
                        print(f"🗎 Custom stages saved to config_param/custom_stages.json: {_merged}")
                except Exception as _e:
                    with save_files_out:
                        print(f"⚠️ Could not write custom_stages.json: {_e}")

            # names registry (config_param/file_names.json): the suffix read and the suffix written, followed by
            # the hypnogram pre-choice of the later tools (and of this tool at its next run)
            if saved and STATE.folder_path:
                _values = {'hypnogram_remapped': out_suffix}
                if STATE.hypno_suffix:
                    _values['hypnogram_scored'] = STATE.hypno_suffix
                _names_warn = SF.write_names(STATE.folder_path, '3', _values)
                if _names_warn:
                    with save_files_out:
                        print(f"⚠️ {_names_warn}")

            invalidate_after('save')

            if saved:
                _custom_note = ''
                if _present_custom:
                    _labels_html = ', '.join(f'<code>{s}</code>' for s in _present_custom)
                    _custom_note = (
                        f'<p>ℹ️ Custom (non-AASM) stage(s) {_labels_html} were kept and saved to '
                        '<code>config_param/custom_stages.json</code>: tools 6, 7 and 8 will '
                        'automatically recognise them.</p>'
                    )
                with out_end:
                    out_end.clear_output()
                    display(HTML(UI.box('ok',
                        '<b>Done!</b>'
                        '<p>Your remapped hypnograms are ready for downstream analyses '
                        '(spectral analysis, YASA, statistics…).</p>'
                        f'{_custom_note}'
                        f'<p><b>Files saved with suffix:</b> <code>{out_suffix}</code></p>'
                        '<p><b>To load a remapped hypnogram in Python:</b></p>'
                        '<pre style="background:#f8f9fa; padding:8px; border-radius:4px;">'
                        'import numpy as np\n'
                        f'hypno = np.loadtxt("participantID{out_suffix}", dtype=str)'
                        '</pre>'
                    )))

        btn_preview.on_click(on_preview)
        btn_save_files.on_click(on_save_files)


#%% --- SECTION 5: VERIFY ---

btn_run_verify = UI.button('Verify', 'run')
STATE.ui['run_verify'] = btn_run_verify


def run_verify(_=None):
    with out_verify:
        out_verify.clear_output()

        folder = STATE.folder_path
        suffix = STATE.output_suffix

        if not folder or not suffix:
            print("⚠️ Please complete the previous steps first.")
            return

        saved_files = detect_hypno_files(folder, suffix)
        if not saved_files:
            print(f"⚠️ No files found with suffix '{suffix}'. Run Section 4 first.")
            return

        print(f"📂 Found {len(saved_files)} remapped file(s) with suffix '{suffix}'.\n")

        configs_after  = {}
        non_aasm_after = {}
        failed_after   = []

        for fpath in saved_files:
            file_id = extract_file_id(fpath, suffix)
            try:
                hypno = np.loadtxt(fpath, dtype=str).astype('<U10')
                key   = frozenset(np.unique(hypno).tolist())
                configs_after.setdefault(key, [])
                configs_after[key].append(file_id)
                non_aasm = [l for l in np.unique(hypno) if l not in current_acceptable()]
                if non_aasm:
                    non_aasm_after[file_id] = non_aasm
            except Exception as e:
                failed_after.append((file_id, str(e)))

        n_before = len(STATE.configs) if STATE.configs is not None else '?'
        print(f"─── Before remapping: {n_before} configuration(s) ───")
        if STATE.configs is not None and STATE.label_configs is not None:
            for i, cfg in enumerate(STATE.configs):
                fids = STATE.label_configs[cfg]
                print(f"  Config. {i+1} (n={len(fids)}): {sorted(cfg)}")

        print(f"\n─── After remapping: {len(configs_after)} configuration(s) ───")
        for i, (cfg, fids) in enumerate(configs_after.items()):
            non_std = [l for l in sorted(cfg) if l not in current_acceptable()]
            note    = f"  ⚠️ non-AASM: {non_std}" if non_std else ""
            print(f"  Config. {i+1} (n={len(fids)}): {sorted(cfg)}{note}")
        if len(configs_after) > 1:
            print("  (Multiple configurations are normal when some participants "
                  "are missing certain sleep stages, e.g. no N3 in insomnia patients.)")

        if not non_aasm_after:
            present_custom = sorted({l for cfg in configs_after for l in cfg
                                     if l not in ACCEPTABLE_LABELS})
            extra = f" + custom: {present_custom}" if present_custom else ""
            print("\n✅ Remapping successful! All files contain only accepted "
                  f"labels (W, N1, N2, N3, R, MT{extra}).")
        else:
            print(f"\n❌ {len(non_aasm_after)} file(s) still contain non-AASM labels. "
                  "Review your mapping rules in Section 3.")
            for fid, labels in non_aasm_after.items():
                print(f"  {fid}: {labels}")

        if failed_after:
            print(f"\n❌ Could not reload {len(failed_after)} file(s):")
            for fid, e in failed_after:
                print(f"  {fid}: {e}")


btn_run_verify.on_click(run_verify)

#%% --- MAIN LAYOUT ---

ui_layout = widgets.VBox([
    section1,
    chooser,
    suffix_menu,
    suffix_box,
    suffix_info,
    exclude_existing_check,
    custom_stages_box,
    custom_stages_info,
    advanced,
    widgets.HBox([btn_scan]),
    scan_progress_row,
    out_scan,
    out_configs,
    section2,
    out_tsv_info,
    out_review,
    out_proceed_s2,
    out_proceed_s2_msg,
    section3,
    out_remap,
    section4,
    out_save,
    section5,
    out_verify,
    out_end,
])

for _out in (out_review, out_remap, out_save, out_verify):
    _show_waiting(_out, 1)
display(ui_layout)

<hr style="height:4px; background-color:black; border:none;">

## References

- Troester MM, Quan SF, Berry RB, et al. The AASM manual for the scoring of sleep and associated events: rules,
  terminology and technical specifications. Version 3. Darien (IL): American Academy of Sleep Medicine; 2023
  (the stages `W`, `N1`, `N2`, `N3`, `R`).
- Rechtschaffen A, Kales A. A manual of standardized terminology, techniques and scoring system for sleep
  stages of human subjects. Washington (DC): Public Health Service, US Government Printing Office; 1968 (the former stages 1 to 4: stages 3
  and 4 are merged into `N3`).